### Maintenance Alert Logic: Machine Learning (ML) Based Approach

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

plant1_generation_path = 'Plant_1_Generation_Data.csv'
plant2_generation_path = 'Plant_2_Generation_Data.csv'
plant1_weather_path = 'Plant_1_Weather_Sensor_Data.csv'
plant2_weather_path = 'Plant_2_Weather_Sensor_Data.csv'

try:
    df_gen1 = pd.read_csv(plant1_generation_path)
    df_gen1['DATE_TIME'] = pd.to_datetime(df_gen1['DATE_TIME'], errors='coerce', dayfirst=True)
    df_gen1.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError:
    df_gen1 = pd.DataFrame()

try:
    df_weather1 = pd.read_csv(plant1_weather_path)
    df_weather1['DATE_TIME'] = pd.to_datetime(df_weather1['DATE_TIME'], errors='coerce')
    df_weather1.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError:
    df_weather1 = pd.DataFrame()

try:
    df_gen2 = pd.read_csv(plant2_generation_path)
    df_gen2['DATE_TIME'] = pd.to_datetime(df_gen2['DATE_TIME'], errors='coerce')
    df_gen2.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError:
    df_gen2 = pd.DataFrame()

try:
    df_weather2 = pd.read_csv(plant2_weather_path)
    df_weather2['DATE_TIME'] = pd.to_datetime(df_weather2['DATE_TIME'], errors='coerce')
    df_weather2.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError:
    df_weather2 = pd.DataFrame()

def clean_dataframe(df):
    numeric_cols = df.select_dtypes(include=np.number).columns
    if not numeric_cols.empty:
        df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both')
        df[numeric_cols] = df[numeric_cols].ffill().bfill()
    return df

if not df_gen1.empty:
    df_gen1 = clean_dataframe(df_gen1.copy())
if not df_weather1.empty:
    df_weather1 = clean_dataframe(df_weather1.copy())
if not df_gen2.empty:
    df_gen2 = clean_dataframe(df_gen2.copy())
if not df_weather2.empty:
    df_weather2 = clean_dataframe(df_weather2.copy())

merged_plant1_data = pd.merge(
    df_gen1[['DATE_TIME', 'PLANT_ID', 'DC_POWER', 'AC_POWER', 'TOTAL_YIELD']],
    df_weather1[['DATE_TIME', 'PLANT_ID', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']],
    on=['DATE_TIME', 'PLANT_ID'],
    how='inner'
)
merged_plant1_data['INVERTER_EFFICIENCY'] = np.where(
    merged_plant1_data['DC_POWER'] > 0,
    merged_plant1_data['AC_POWER'] / merged_plant1_data['DC_POWER'],
    0
).clip(max=1.0)

merged_plant2_data = pd.merge(
    df_gen2[['DATE_TIME', 'PLANT_ID', 'DC_POWER', 'AC_POWER', 'TOTAL_YIELD']],
    df_weather2[['DATE_TIME', 'PLANT_ID', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']],
    on=['DATE_TIME', 'PLANT_ID'],
    how='inner'
)
merged_plant2_data['INVERTER_EFFICIENCY'] = np.where(
    merged_plant2_data['DC_POWER'] > 0,
    merged_plant2_data['AC_POWER'] / merged_plant2_data['DC_POWER'],
    0
).clip(max=1.0)

plant1_power_per_irr = merged_plant1_data[
    (merged_plant1_data['IRRADIATION'] > 0) & (merged_plant1_data['DC_POWER'] > 0)
].copy()

if not plant1_power_per_irr.empty:
    plant1_power_per_irr['DC_POWER_PER_IRRADIATION'] = plant1_power_per_irr['DC_POWER'] / plant1_power_per_irr['IRRADIATION']
    max_dc_per_irr_plant1 = plant1_power_per_irr['DC_POWER_PER_IRRADIATION'].max()
    merged_plant1_data['IDEAL_DC_POWER'] = merged_plant1_data['IRRADIATION'] * max_dc_per_irr_plant1
else:
    merged_plant1_data['IDEAL_DC_POWER'] = 0

plant2_power_per_irr = merged_plant2_data[
    (merged_plant2_data['IRRADIATION'] > 0) & (merged_plant2_data['DC_POWER'] > 0)
].copy()

if not plant2_power_per_irr.empty:
    plant2_power_per_irr['DC_POWER_PER_IRRADIATION'] = plant2_power_per_irr['DC_POWER'] / plant2_power_per_irr['IRRADIATION']
    max_dc_per_irr_plant2 = plant2_power_per_irr['DC_POWER_PER_IRRADIATION'].max()
    merged_plant2_data['IDEAL_DC_POWER'] = merged_plant2_data['IRRADIATION'] * max_dc_per_irr_plant2
else:
    merged_plant2_data['IDEAL_DC_POWER'] = 0

def add_weather_noise(df, irradiation_noise_std=0.05, temp_noise_std=2.0, seed=42):
    np.random.seed(seed)
    noisy_df = df.copy()
    if 'IRRADIATION' in noisy_df.columns:
        noise = np.random.normal(0, irradiation_noise_std, size=len(noisy_df))
        noisy_df['IRRADIATION'] = noisy_df['IRRADIATION'] + noise
        noisy_df['IRRADIATION'] = noisy_df['IRRADIATION'].clip(lower=0)
    if 'AMBIENT_TEMPERATURE' in noisy_df.columns:
        noise = np.random.normal(0, temp_noise_std, size=len(noisy_df))
        noisy_df['AMBIENT_TEMPERATURE'] = noisy_df['AMBIENT_TEMPERATURE'] + noise
    if 'MODULE_TEMPERATURE' in noisy_df.columns:
        noise = np.random.normal(0, temp_noise_std, size=len(noisy_df))
        noisy_df['MODULE_TEMPERATURE'] = noisy_df['MODULE_TEMPERATURE'] + noise
    return noisy_df

noisy_plant1_data = add_weather_noise(merged_plant1_data.copy(), irradiation_noise_std=0.01, temp_noise_std=1.0)
noisy_plant1_data['INVERTER_EFFICIENCY'] = np.where(
    noisy_plant1_data['DC_POWER'] > 0,
    noisy_plant1_data['AC_POWER'] / noisy_plant1_data['DC_POWER'],
    0
).clip(max=1.0)
noisy_plant1_data['IDEAL_DC_POWER'] = noisy_plant1_data['IRRADIATION'] * max_dc_per_irr_plant1

noisy_plant2_data = add_weather_noise(merged_plant2_data.copy(), irradiation_noise_std=0.005, temp_noise_std=0.5)
noisy_plant2_data['INVERTER_EFFICIENCY'] = np.where(
    noisy_plant2_data['DC_POWER'] > 0,
    noisy_plant2_data['AC_POWER'] / noisy_plant2_data['DC_POWER'],
    0
).clip(max=1.0)
noisy_plant2_data['IDEAL_DC_POWER'] = noisy_plant2_data['IRRADIATION'] * max_dc_per_irr_plant2

X_plant1 = merged_plant1_data[['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']]
y_plant1 = merged_plant1_data['DC_POWER']
X_plant1 = X_plant1.dropna()
y_plant1 = y_plant1[X_plant1.index]
X_train_plant1, X_test_plant1, y_train_plant1, y_test_plant1 = train_test_split(X_plant1, y_plant1, test_size=0.2, random_state=42)

X_plant2 = merged_plant2_data[['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']]
y_plant2 = merged_plant2_data['DC_POWER']
X_plant2 = X_plant2.dropna()
y_plant2 = y_plant2[X_plant2.index]
X_train_plant2, X_test_plant2, y_train_plant2, y_test_plant2 = train_test_split(X_plant2, y_plant2, test_size=0.2, random_state=42)

model_plant1 = LinearRegression()
model_plant1.fit(X_train_plant1, y_train_plant1)

y_pred_plant1 = model_plant1.predict(X_test_plant1)
r2_plant1 = r2_score(y_test_plant1, y_pred_plant1)

model_plant2 = LinearRegression()
model_plant2.fit(X_train_plant2, y_train_plant2)

y_pred_plant2 = model_plant2.predict(X_test_plant2)
r2_plant2 = r2_score(y_test_plant2, y_pred_plant2)

In [ ]:
def generate_ml_maintenance_alerts(df, model, features, prediction_threshold_ratio=0.85):
    """
    Generates maintenance alerts based on a machine learning model's prediction of DC Power.

    Args:
        df (pd.DataFrame): The DataFrame containing 'DC_POWER' and the 'features' used by the model.
        model (sklearn.linear_model.LinearRegression): The trained linear regression model.
        features (list): A list of column names in df that correspond to the model's features.
        prediction_threshold_ratio (float): The ratio of actual DC_POWER to predicted DC_POWER.
                                        An alert is triggered if DC_POWER < prediction_threshold_ratio * predicted_DC_POWER.

    Returns:
        pd.DataFrame: The DataFrame with an 'ML_ALERT' column indicating maintenance alerts.
    """
    df_ml_alerts = df.copy()
    df_ml_alerts['ML_ALERT'] = False

    # Only make predictions where IRRADIATION is positive, as DC_POWER is 0 when no light
    active_generation_indices = df_ml_alerts[df_ml_alerts['IRRADIATION'] > 0].index

    if not active_generation_indices.empty:
        # Prepare features for prediction
        X_predict = df_ml_alerts.loc[active_generation_indices, features]

        # Predict expected DC_POWER
        df_ml_alerts.loc[active_generation_indices, 'PREDICTED_DC_POWER'] = model.predict(X_predict)

        # Rule: Actual DC_POWER significantly lower than predicted DC_POWER
        condition_ml_alert = ((df_ml_alerts['DC_POWER'] < prediction_threshold_ratio * df_ml_alerts['PREDICTED_DC_POWER']) &
                             (df_ml_alerts['DC_POWER'] > 0))
        df_ml_alerts.loc[condition_ml_alert, 'ML_ALERT'] = True
    else:
        df_ml_alerts['PREDICTED_DC_POWER'] = 0

    return df_ml_alerts

# Define the features used by the models
model_features = ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']

# --- Apply ML alert logic to Plant 1 data (original) ---
ml_alerts_original_plant1 = generate_ml_maintenance_alerts(merged_plant1_data.copy(), model_plant1, model_features)

# --- Apply ML alert logic to Plant 2 data (original) ---
ml_alerts_original_plant2 = generate_ml_maintenance_alerts(merged_plant2_data.copy(), model_plant2, model_features)

# --- Apply ML alert logic to Plant 1 data (noisy) ---
ml_alerts_noisy_plant1 = generate_ml_maintenance_alerts(noisy_plant1_data.copy(), model_plant1, model_features)

# --- Apply ML alert logic to Plant 2 data (noisy) ---
ml_alerts_noisy_plant2 = generate_ml_maintenance_alerts(noisy_plant2_data.copy(), model_plant2, model_features)

In [ ]:
print("\n--- Evaluating ML Alert Accuracy ---")

# --- Plant 1 ML Alert Evaluation ---
print("\n--- Plant 1 ML Alert Comparison ---")
original_ml_alerts_plant1 = ml_alerts_original_plant1[ml_alerts_original_plant1['ML_ALERT'] == True]
noisy_ml_alerts_plant1 = ml_alerts_noisy_plant1[ml_alerts_noisy_plant1['ML_ALERT'] == True]

print(f"Total ML alerts on original Plant 1 data: {len(original_ml_alerts_plant1)}")
print(f"Total ML alerts on noisy Plant 1 data: {len(noisy_ml_alerts_plant1)}")

# Find common alerts (DATE_TIME must match)
common_ml_alerts_plant1 = pd.merge(original_ml_alerts_plant1[['DATE_TIME']], noisy_ml_alerts_plant1[['DATE_TIME']], on='DATE_TIME', how='inner')
print(f"Common ML alerts (triggered in both original and noisy) for Plant 1: {len(common_ml_alerts_plant1)}")

# Alerts only in noisy data (potential false positives due to noise)
noisy_only_ml_alerts_plant1 = noisy_ml_alerts_plant1[~noisy_ml_alerts_plant1['DATE_TIME'].isin(original_ml_alerts_plant1['DATE_TIME'])]
print(f"ML alerts only in noisy data for Plant 1 (potential noise-induced false positives): {len(noisy_only_ml_alerts_plant1)}")

# Alerts only in original data (potential false negatives due to noise masking issues)
original_only_ml_alerts_plant1 = original_ml_alerts_plant1[~original_ml_alerts_plant1['DATE_TIME'].isin(noisy_ml_alerts_plant1['DATE_TIME'])]
print(f"ML alerts only in original data for Plant 1 (potential noise-masked true positives): {len(original_only_ml_alerts_plant1)}")

# --- Plant 2 ML Alert Evaluation ---
print("\n--- Plant 2 ML Alert Comparison ---")
original_ml_alerts_plant2 = ml_alerts_original_plant2[ml_alerts_original_plant2['ML_ALERT'] == True]
noisy_ml_alerts_plant2 = ml_alerts_noisy_plant2[ml_alerts_noisy_plant2['ML_ALERT'] == True]

print(f"Total ML alerts on original Plant 2 data: {len(original_ml_alerts_plant2)}")
print(f"Total ML alerts on noisy Plant 2 data: {len(noisy_ml_alerts_plant2)}")

# Find common alerts
common_ml_alerts_plant2 = pd.merge(original_ml_alerts_plant2[['DATE_TIME']], noisy_ml_alerts_plant2[['DATE_TIME']], on='DATE_TIME', how='inner')
print(f"Common ML alerts (triggered in both original and noisy) for Plant 2: {len(common_ml_alerts_plant2)}")

# Alerts only in noisy data (potential false positives due to noise)
noisy_only_ml_alerts_plant2 = noisy_ml_alerts_plant2[~noisy_ml_alerts_plant2['DATE_TIME'].isin(original_ml_alerts_plant2['DATE_TIME'])]
print(f"ML alerts only in noisy data for Plant 2 (potential noise-induced false positives): {len(noisy_only_ml_alerts_plant2)}")

# Alerts only in original data (potential false negatives due to noise masking issues)
original_only_ml_alerts_plant2 = original_ml_alerts_plant2[~original_ml_alerts_plant2['DATE_TIME'].isin(noisy_ml_alerts_plant2['DATE_TIME'])]
print(f"ML alerts only in original data for Plant 2 (potential noise-masked true positives): {len(original_only_ml_alerts_plant2)}")